# CeNNMixer-v3 — Cellular Mixer + Global Recurrent Memory

v3 targets the main v2 limitation: v2 could preserve quality around alpha 0.10–0.25, but degraded near alpha 0.50.

CeNNMixer-v3 keeps:
- fast / mid / slow grouped cellular recurrent state
- sparse group-neighbor communication
- progressive Qwen-mixer takeover

and adds:
- a small fixed recurrent global memory
- content-based slot read/write/erase
- top-k slot routing
- no token-to-token softmax attention
- fixed memory cost per token

Default memory:

    8 slots × 64 dimensions
    top-4 slot routing

Quick mode uses a finer takeover curriculum:

    0 → .10 → .20 → .30 → .40 → .50 → .60 → .75 → 1.0

The main question for this notebook is whether v3 can pass the v2 failure region around alpha 0.5.


In [ ]:
#@title 1. Setup
import pathlib, subprocess, sys, importlib, json, torch, shutil

REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers','accelerate','datasets','pandas','matplotlib',
    'huggingface_hub','safetensors'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC=REPO_DIR/'src'
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))
for n in list(sys.modules):
    if n=='tinycenn_lm' or n.startswith('tinycenn_lm.'):
        del sys.modules[n]
importlib.invalidate_caches()

for p in [
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v3_core.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v3.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v3_train.py',
    REPO_DIR/'scripts'/'run_qwen35_cennmixer_v3.py',
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

print('✓ CeNNMixer-v3 preflight OK')
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
else:
    print('⚠️ Select a GPU runtime.')


In [ ]:
#@title 2. Configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
LAYERS='0' #@param {type:'string'}
QUICK_SMOKE=True #@param {type:'boolean'}

GROUPS=32 #@param {type:'integer'}
CELL_DIM=48 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}

MEMORY_SLOTS=8 #@param {type:'integer'}
MEMORY_DIM=64 #@param {type:'integer'}
MEMORY_TOPK=4 #@param {type:'integer'}

LR=0.0002 #@param {type:'number'}

# Used when QUICK_SMOKE=False.
ALPHAS='0,0.05,0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90,1.0' #@param {type:'string'}
SEQ_LEN=128 #@param {type:'integer'}
TRAIN_BLOCKS=1024 #@param {type:'integer'}
VAL_BLOCKS=48 #@param {type:'integer'}
STAGE_UPDATES=300 #@param {type:'integer'}
EXTEND_UPDATES=200 #@param {type:'integer'}
MAX_STAGE_UPDATES=3000 #@param {type:'integer'}
PROBE_EVERY=50 #@param {type:'integer'}
PATIENCE_PROBES=8 #@param {type:'integer'}
MIN_LR=0.0000125 #@param {type:'number'}
TOPK=64 #@param {type:'integer'}

MIN_TOP1=0.97 #@param {type:'number'}
MAX_KL=0.03 #@param {type:'number'}
MAX_HIDDEN_MSE=0.05 #@param {type:'number'}
MAX_MIXER_MSE=0.12 #@param {type:'number'}

OUTPUT_DIR=REPO_DIR/'results'/'cennmixer_v3_qwen35_08b'

print('Quick smoke:',QUICK_SMOKE)
print('Layers:',LAYERS)
print('Cellular state:',GROUPS,'×',CELL_DIM,'per timescale')
print('Global memory:',MEMORY_SLOTS,'×',MEMORY_DIM,'top-k',MEMORY_TOPK)


### v3 quick-mode settings

With `QUICK_SMOKE=True`, the runner overrides only the training budget/data curriculum:

    alpha: 0 → .10 → .20 → .30 → .40 → .50 → .60 → .75 → 1.0
    seq_len: 64
    train_blocks: 256
    validation blocks: 16
    probe every: 20

The actual v3 architecture remains the configured 32×48 cellular state plus 8×64 global recurrent memory.

The critical comparison with v2 is alpha=0.50:
- v2 best top-1 was ~0.80
- v3 should ideally keep substantially better agreement and generation similarity there.


In [ ]:
#@title 3. Train CeNNMixer-v3
cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_cennmixer_v3.py'),
    '--base-model',BASE_MODEL,
    '--layers',LAYERS,
    '--alphas',ALPHAS,
    '--seq-len',str(SEQ_LEN),
    '--train-blocks',str(TRAIN_BLOCKS),
    '--val-blocks',str(VAL_BLOCKS),
    '--groups',str(GROUPS),
    '--cell-dim',str(CELL_DIM),
    '--graph-steps',str(GRAPH_STEPS),
    '--memory-slots',str(MEMORY_SLOTS),
    '--memory-dim',str(MEMORY_DIM),
    '--memory-topk',str(MEMORY_TOPK),
    '--lr',str(LR),
    '--stage-updates',str(STAGE_UPDATES),
    '--extend-updates',str(EXTEND_UPDATES),
    '--max-stage-updates',str(MAX_STAGE_UPDATES),
    '--probe-every',str(PROBE_EVERY),
    '--patience-probes',str(PATIENCE_PROBES),
    '--min-lr',str(MIN_LR),
    '--topk',str(TOPK),
    '--min-top1',str(MIN_TOP1),
    '--max-kl',str(MAX_KL),
    '--max-hidden-mse',str(MAX_HIDDEN_MSE),
    '--max-mixer-mse',str(MAX_MIXER_MSE),
    '--output-dir',str(OUTPUT_DIR),
]
if QUICK_SMOKE:
    cmd.append('--quick-smoke')

print('='*120)
print(' '.join(cmd))
print('='*120)
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
print('\nFinished, exit code',rc)
if rc:
    raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Stage summary
import pandas as pd
from IPython.display import display

report=json.loads((OUTPUT_DIR/'report.json').read_text())
stage=pd.read_csv(OUTPUT_DIR/'stage_summary.csv')
hist=pd.read_csv(OUTPUT_DIR/'training_history.csv')

print('Architecture:',report['architecture'])
print('Reached alpha:',report['reached_alpha'])
print('Progression complete:',report['progression_complete'])
print('Strict final quality:',report['strict_quality_gate'])
print('CeNN-v3 params:',f"{report['cenn_params']:,}")
print('Qwen mixer params:',f"{report['replaced_qwen_mixer_params']:,}")
print('Mixer parameter reduction:',f"{report['mixer_param_reduction_pct']:.2f}%")
print('Memory config:',{
    'slots':report['config']['memory_slots'],
    'dim':report['config']['memory_dim'],
    'topk':report['config']['memory_topk'],
})

cols=[
    'alpha','best_step','trained_steps','pass','violation',
    'student_ce','ce_gap','kl','hidden_mse','delta_mse',
    'mixer_mse','mixer_cosine','mixer_delta','top1',
    'generation_exact_rate','generation_mean_jaccard'
]
display(stage[cols])


In [ ]:
#@title 5. Takeover curves
import matplotlib.pyplot as plt

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['top1'],marker='o')
plt.axhline(0.80,linestyle='--')
plt.xlabel('alpha')
plt.ylabel('top-1 agreement')
plt.title('CeNNMixer-v3 progressive takeover')
plt.show()

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['kl'],marker='o')
plt.xlabel('alpha')
plt.ylabel('KL')
plt.title('Output-distribution drift')
plt.show()

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['mixer_mse'],marker='o',label='mixer MSE')
plt.plot(stage['alpha'],stage['mixer_cosine'],marker='o',label='cosine error')
plt.plot(stage['alpha'],stage['mixer_delta'],marker='o',label='temporal error')
plt.xlabel('alpha')
plt.ylabel('error')
plt.legend()
plt.title('Local Qwen mixer → CeNN-v3 fidelity')
plt.show()


In [ ]:
#@title 6. Inspect alpha=0.5 specifically
half=stage[(stage['alpha']-0.5).abs()<1e-8]
if len(half):
    display(half[[
        'alpha','top1','kl','hidden_mse',
        'mixer_mse','mixer_cosine','mixer_delta',
        'generation_exact_rate','generation_mean_jaccard'
    ]])
    print('v2 reference: top1 around 0.797 at alpha=0.5')
else:
    print('The run stopped before alpha=0.5. Inspect the previous stage readiness metrics.')


In [ ]:
#@title 7. Final generation results
if report.get('progression_complete'):
    print('CeNN-only generation after removing the Qwen mixer:')
    rows=report['final_cenn_only_generation']
else:
    last=float(stage.iloc[-1]['alpha'])
    path=OUTPUT_DIR/f"generation_alpha_{str(last).replace('.','p')}.json"
    rows=json.loads(path.read_text())
    print('Progression stopped at alpha',last,'— showing best generation at that stage.')

for i,x in enumerate(rows,1):
    print('\n'+'='*100)
    print(i,'USER:',x['prompt'])
    print('QWEN:',x['qwen'])
    print('CeNN-v3:',x['cenn'])
    print('exact=',x['exact'],'jaccard=',round(x['jaccard'],3))


In [ ]:
#@title 8. Upload final strict-quality CeNNMixer-v3 to Hugging Face
from huggingface_hub import HfApi, login
import shutil, json

if not report.get('progression_complete'):
    raise RuntimeError('CeNNMixer-v3 has not safely reached alpha=1; HF upload is blocked.')
if not report.get('strict_quality_gate'):
    raise RuntimeError('CeNN-v3 reached alpha=1 but did not pass the strict quality gate; HF upload is blocked.')

HF_REPO_ID='vtava/Qwen35-0.8B-CeNNMixer-v3' #@param {type:'string'}
HF_PRIVATE=False #@param {type:'boolean'}

HF_EXPORT=REPO_DIR/'results'/'Qwen35-0.8B-CeNNMixer-v3-HF'
if HF_EXPORT.exists():
    shutil.rmtree(HF_EXPORT)
HF_EXPORT.mkdir(parents=True,exist_ok=True)

for name in ['cennmixer_v3_final_cenn_only.pt','report.json','stage_summary.csv','training_history.csv']:
    src=OUTPUT_DIR/name
    if src.exists():
        shutil.copy2(src,HF_EXPORT/name)

(HF_EXPORT/'cennmixer_v3_config.json').write_text(json.dumps({
    'base_model':report['base_model'],
    'architecture':report['architecture'],
    'layers':report['layers'],
    'layer_kinds':report['layer_kinds'],
    'config':report['config'],
    'final_metrics':report['final_cenn_only_probe'],
},indent=2),encoding='utf-8')

m=report['final_cenn_only_probe']
readme=f"""---
base_model: {report['base_model']}
library_name: transformers
pipeline_tag: text-generation
tags:
- qwen
- cenn
- recurrent-memory
- sequence-mixer
- tinycenn
---

# Qwen3.5-0.8B CeNNMixer-v3

CeNNMixer-v3 combines grouped multi-timescale cellular recurrent state with
a fixed content-addressable recurrent memory bank.

Layers replaced: {report['layers']}

Memory:
- slots: {report['config']['memory_slots']}
- dimension: {report['config']['memory_dim']}
- routed slots/token: {report['config']['memory_topk']}

Final CeNN-only metrics:
- top-1 agreement: {m['top1']:.6f}
- KL: {m['kl']:.6f}
- hidden MSE: {m['hidden_mse']:.6f}
- student CE: {m['student_ce']:.6f}
- teacher CE: {m['teacher_ce']:.6f}

Mixer parameter reduction: {report['mixer_param_reduction_pct']:.2f}%

Project: https://github.com/vtavakkoli/TinyCeNN-LM
"""
(HF_EXPORT/'README.md').write_text(readme,encoding='utf-8')

token=None
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
except Exception:
    pass
if token:
    login(token=token,add_to_git_credential=False)
else:
    login()

api=HfApi()
api.create_repo(HF_REPO_ID,repo_type='model',private=HF_PRIVATE,exist_ok=True)
api.upload_folder(
    folder_path=str(HF_EXPORT),
    repo_id=HF_REPO_ID,
    repo_type='model',
    commit_message='Upload CeNNMixer-v3 global-memory adapter and results',
)
print('✓ Uploaded: https://huggingface.co/'+HF_REPO_ID)
